### 馬達論文研究
### 第三步 遷移式學習
### 馬達A 8000RPM
### CNN_VGG
### 雙階段泛化

In [ ]:
# --- logging bootstrap (auto-added) ---
import importlib
from scripts import logger as _logger_mod
_logger_mod = importlib.reload(_logger_mod)
save_plot = _logger_mod.save_plot
setup_logger = _logger_mod.setup_logger
tee_std_to_file = _logger_mod.tee_std_to_file

LOG, RUN_PATHS = setup_logger('notebook', console=False)
_tee_ctx = tee_std_to_file(RUN_PATHS.log_file)
_tee_ctx.__enter__()
import atexit
atexit.register(_tee_ctx.__exit__, None, None, None)

# Ensure TensorFlow releases GPU/graph resources on exit
try:
    import tensorflow as tf
    atexit.register(tf.keras.backend.clear_session)
except Exception:
    pass

# Auto-save matplotlib figures on plt.show()
try:
    import matplotlib.pyplot as plt
    if not getattr(plt, '_ancestor_save_plot_patched', False):
        plt._ancestor_save_plot_patched = True
        _orig_show = plt.show
        import time
        plt._ancestor_show_in_progress = False
        plt._ancestor_last_save_ts = 0.0

        def _show_and_save(*args, **kwargs):
            if getattr(plt, '_ancestor_show_in_progress', False):
                return _orig_show(*args, **kwargs)
            now = time.monotonic()
            if now - float(getattr(plt, '_ancestor_last_save_ts', 0.0)) < 0.5:
                return _orig_show(*args, **kwargs)
            plt._ancestor_show_in_progress = True
            try:
                save_plot(plt, LOG, RUN_PATHS)
            except Exception:
                pass
            try:
                return _orig_show(*args, **kwargs)
            finally:
                plt._ancestor_last_save_ts = time.monotonic()
                try:
                    plt.close(plt.gcf())
                except Exception:
                    pass
                plt._ancestor_show_in_progress = False

        plt.show = _show_and_save
except Exception:
    pass
# --- end logging bootstrap ---


In [ ]:
# 匯入所需函式庫
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
from scripts.gpu_utils import device_scope, DEVICE
warnings.filterwarnings("ignore")

In [ ]:
# 資料準備與處理
def load_and_preprocess_data(base_dir, screws_list):
    datasets = []
    for screws in screws_list:
        file_path = os.path.join(base_dir, 'T3', '8000rpm', screws, 'T3_Group_feature_data.csv')
        if os.path.exists(file_path):
            dataset = pd.read_csv(file_path)
            dataset['screws'] = screws
            datasets.append(dataset)
        else:
            print(f"檔案未找到: {file_path}")
    return datasets

In [ ]:
# 加載資料
rootDir = os.getcwd()
modelDirectory = os.path.join(rootDir, 'data', 'Step-1', 'model')myfeatureDirectory = os.path.join(rootDir, 'data', 'Step-3', 'myfeature')
screws_list = ['8screws', '1screws', '2screws', '3screws', '4screws']
datasets_T3 = load_and_preprocess_data(myfeatureDirectory, screws_list)

In [ ]:
# 合併資料
combined_data = pd.concat(datasets_T3, ignore_index=True)

In [ ]:
# **手動定義標籤順序**
desired_order = ['8screws', '1screws', '2screws', '3screws', '4screws']
label_mapping = {screw: idx for idx, screw in enumerate(desired_order)}
combined_data['screws'] = combined_data['screws'].map(label_mapping)

In [ ]:
# 標籤編碼
le_screws = LabelEncoder()
combined_data['screws'] = le_screws.fit_transform(combined_data['screws'])

In [ ]:
# 確認標籤映射是否正確
print("自訂 Label Mapping:")
for label, num in label_mapping.items():
    print(f"{label}: {num}")

In [ ]:
# 資料分割
X = combined_data.drop(['screws'], axis=1).values
y_screws = combined_data['screws'].values

In [ ]:
# 劃分訓練與測試集
X_train, X_test, y_train_screws, y_test_screws = train_test_split(
    X, y_screws, test_size=0.2, random_state=42
)

In [ ]:
# 標準化特徵
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 載入模型
model_path = os.path.join(modelDirectory, 'VGG16_C8000.keras')
# 載入模型
model = load_model(model_path)

# 確認模型載入成功
print("模型已成功載入。")

In [ ]:
# 計算需要凍結的層數（保留 50% 可訓練）
num_layers = len(model.layers)
train_layers_1 = int(num_layers * 0.5)  # 50% 可訓練
freeze_layers_1 = num_layers - train_layers_1  # 50% 凍結

# 凍結前 freeze_layers層
for layer in model.layers[:freeze_layers_1]:
    layer.trainable = False

# 確認層狀態
for i, layer in enumerate(model.layers):
    print(f"Model 2 - Layer {i}: {layer.name} | Trainable: {layer.trainable}")

In [ ]:
# 重新訓練模型
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=["sparse_categorical_crossentropy"],
    metrics=["accuracy"]
)

In [ ]:
print(f"[Training] Device: {DEVICE}")
with device_scope():
    # 訓練模型
    history = model.fit(
        X_train.reshape(X_train.shape[0], X_train.shape[1], 1),
        y_train_screws,
        validation_data=(
            X_test.reshape(X_test.shape[0], X_test.shape[1], 1),
            y_test_screws
        ),
        epochs=100,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
    )

In [ ]:
# 清除
tf.keras.backend.clear_session()

In [ ]:
# 模型架構
model.summary()

In [ ]:
# 評估模型
results = model.evaluate(
    X_test.reshape(X_test.shape[0], X_test.shape[1], 1),
    y_test_screws
)
print(f"Loss: {results[0]:.4f}, Accuracy: {results[1]:.4f}")

In [ ]:
# 繪製學習曲線
plt.figure()
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Learning Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Learning Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# 混淆矩陣繪製
def plot_confusion_matrix(y_true, y_pred, label_encoder, title):
    """
    繪製混淆矩陣，顯示數字映射標籤
    """
    cm = confusion_matrix(y_true, y_pred)
    numeric_labels = range(len(label_encoder.classes_))  # 使用數字標籤
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=numeric_labels, yticklabels=numeric_labels)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

# 預測並繪製混淆矩陣
y_pred_screw = model.predict(X_test.reshape(X_test.shape[0], X_test.shape[1], 1)).argmax(axis=1)
plot_confusion_matrix(y_test_screws, y_pred_screw, le_screws, "Confusion Matrix - Model 21 - Two-stage generalization")

In [ ]:
# 儲存模型
model.save(os.path.join(rootDir, 'data', 'Step-3', 'model', 'VGG16_A8000.keras'))